# 04 — PPO training

The default smoke path trains against a mock victim and writes `outputs/checkpoints/ppo_notebook_smoke.zip`. Switch `USE_MOCK` off and configure the paths to train against DeepfakeBench Xception.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")

In [ ]:
USE_MOCK = True
TRAIN_DIR = PROJECT_ROOT / "data/train"
DEEPFAKEBENCH_ROOT = PROJECT_ROOT.parent / "DeepfakeBench"
XCEPTION_WEIGHTS = DEEPFAKEBENCH_ROOT / "training/weights/xception_best.pth"
TIMESTEPS = 32 if USE_MOCK else 10_000
SEED = 42
DEVICE = "auto"


In [ ]:
if USE_MOCK:
    import numpy as np
    from PIL import Image
    from ppo.agent import create_ppo_agent
    from ppo.environment import DeepfakeAttackEnv
    from src.ppo.mock_detector import MockDetector

    demo_dir = PROJECT_ROOT / "outputs/notebook_demo/train"
    demo_dir.mkdir(parents=True, exist_ok=True)
    paths = []
    for index, level in enumerate((160, 190, 220)):
        path = demo_dir / f"synthetic_{index}.png"
        Image.fromarray(np.full((64, 64, 3), level, dtype=np.uint8)).save(path)
        paths.append(path)
    env = DeepfakeAttackEnv(paths, MockDetector(), max_steps=3, seed=SEED)
    model = create_ppo_agent(env, seed=SEED, device=DEVICE, n_steps=16, batch_size=8, verbose=0)
    model.learn(total_timesteps=TIMESTEPS)
    checkpoint = PROJECT_ROOT / "outputs/checkpoints/ppo_notebook_smoke"
    checkpoint.parent.mkdir(parents=True, exist_ok=True)
    model.save(checkpoint)
    print(checkpoint.with_suffix(".zip"))
else:
    from ppo.train import train_attack
    checkpoint = train_attack(
        train_dir=TRAIN_DIR,
        deepfakebench_root=DEEPFAKEBENCH_ROOT,
        weights_path=XCEPTION_WEIGHTS,
        timesteps=TIMESTEPS,
        seed=SEED,
        device=DEVICE,
        output_dir=PROJECT_ROOT / "outputs",
    )
    print(checkpoint)

In [ ]:
observation, _ = env.reset(seed=SEED) if USE_MOCK else (None, None)
if USE_MOCK:
    action, _ = model.predict(observation, deterministic=True)
    print("deterministic action [type, strength]:", action)
    assert env.action_space.contains(action)